# Mage-Flow-Edit-Turbo quality experiment (standalone, not wired into the backend)

**Purpose:** run Microsoft's `microsoft/Mage-Flow-Edit` model (MIT license, 4B params, "structure-aware" instruction-based editing) on your room photo, using the *same* Economical/Mid/Premium tier prompts as the production `app/pipeline/prompts.py` (v5) - same setup as the earlier Qwen-Image-Edit notebook, so all three (SD1.5/Qwen/Mage-Flow) are directly comparable.

**Read before running - real limitations, not swept under the rug:**
- No hosted API exists for this model yet (confirmed on its Hugging Face page: "not deployed by any Inference Provider") - self-hosting is the only option, same as the Qwen notebook.
- Microsoft's own numbers: **~18-20GB peak VRAM at 1024x1024 on an A100.** Free Colab's T4 only has 16GB. This notebook works around that by generating at a **lower `max_size` (768 instead of 1024)** to reduce memory - if you still hit an out-of-memory error, lower `MAX_SIZE` further (e.g. 512) in step 5.
- The install requires building `flash-attn` from source with build isolation off - this is the most likely failure point (torch/CUDA version mismatches are common). Step 2 installs a prebuilt wheel matched to Colab's typical CUDA version if possible; if it fails, expect a long compile (10+ min) or a version-mismatch error you'll need to resolve manually.
- **Runtime > Change runtime type > T4 GPU** must be selected before running any cell below.
- This is a manual, one-off comparison experiment - not wired into the FastAPI backend.

## 1. Environment check

In [ ]:
import sys
import torch

print(f"Python: {sys.version}")
print(f"Torch:  {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram_gb:.1f} GB")
    if vram_gb < 18:
        print("WARNING: below the 18-20GB Microsoft measured at 1024px. This notebook uses")
        print("MAX_SIZE=768 to compensate - lower it further in step 5 if you hit an OOM error.")
else:
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > T4 GPU, then re-run this cell."
    )

## 2. Install dependencies
This is the step most likely to need manual troubleshooting. `flash-attn` has to be installed *after* everything else, with build isolation off, and it must match your installed torch/CUDA versions exactly. If the `flash-attn` line fails, note the torch/CUDA versions printed in step 1 and search https://github.com/Dao-AILab/flash-attention/releases for a matching prebuilt wheel, or let it compile from source (slow but usually works eventually).

In [ ]:
%pip install -q -U "transformers==5.5.*" "diffusers==0.38.*" "pillow==12.3.*"

In [ ]:
%pip install -q "git+https://github.com/microsoft/Mage.git#subdirectory=mage_flow" --no-deps

In [ ]:
%pip install -q setuptools wheel ninja
%pip install -q --no-build-isolation flash-attn==2.8.3

## 3. Load the model
Using `microsoft/Mage-Flow-Edit` (the base instruction-edit checkpoint, not the `-Turbo` 4-step distilled variant) with `steps=30` for full quality. Switch to the `-Turbo` checkpoint with `steps=4` if you want faster iteration at some quality cost - see the commented alternative below.

In [ ]:
from mage_flow import MageFlowPipeline

pipe = MageFlowPipeline.from_pretrained("microsoft/Mage-Flow-Edit", device="cuda")
STEPS = 30

# Faster alternative (uncomment to use instead):
# pipe = MageFlowPipeline.from_pretrained("microsoft/Mage-Flow-Edit-Turbo", device="cuda")
# STEPS = 4

print("Model loaded.")

## 4. Upload your room photo

In [ ]:
from google.colab import files

uploaded = files.upload()
ROOM_PHOTO_PATH = next(iter(uploaded.keys()))

from PIL import Image
Image.open(ROOM_PHOTO_PATH)

## 5. Tier prompts
Same as the Qwen notebook - ported directly from the production `app/pipeline/prompts.py` (v5), so all provider experiments stay comparable against each other and against the live Cloudflare/SD1.5 pipeline.

In [ ]:
MAX_SIZE = 768  # lower to 512 if you hit an out-of-memory error on the T4

TIER_SPECS = {
    "economical": {
        "label": "budget renovation",
        "paint": "plain flat cream and sage-green two-tone paint, cheap and utilitarian looking",
        "flooring": "ordinary matte grey ceramic tile flooring, plain and unpolished, no shine, no gloss",
        "lighting_temp": "cool white practical lighting, 5000-6000K",
        "feature_wall": "no accent wall, no wall mouldings, bare plain walls",
        "ceiling": "plain flat white ceiling, no false ceiling, a simple ceiling fan",
        "materials": "paint only, no premium materials, no wood paneling, no marble",
        "palette": "muted cream, sage green, and grey tones",
        "density": "sparse furniture, about 80 percent of floor space left empty, uncluttered and clean",
        "decor": "one or two potted plants, simple thin plain curtains, one or two plain framed prints",
        "structure_reminder": "",
    },
    "mid": {
        "label": "mid-level renovation",
        "paint": "warm beige walls with olive-green wainscoting panel on the lower half",
        "flooring": "warm wood-look laminate flooring",
        "lighting_temp": "neutral warm lighting, 3500-4000K",
        "feature_wall": "wall mouldings and a few framed art prints",
        "ceiling": "false ceiling with a warm cove lighting strip",
        "materials": "paint, wall mouldings, wainscoting, laminate wood flooring",
        "palette": "warm beige, olive green, and light wood tones",
        "density": "balanced furniture arrangement, tidy and comfortable, not crowded",
        "decor": "an area rug, framed art prints, a few potted plants",
        "structure_reminder": "",
    },
    "premium": {
        "label": "premium luxury renovation",
        "paint": "dark wood panel and marble feature wall with brass trim accents",
        "flooring": "polished Italian marble flooring, reflective and bright",
        "lighting_temp": "warm luxury lighting, 2700-3000K, like a luxury hotel lobby",
        "feature_wall": "marble and dark wood panel wall with brass inlay",
        "ceiling": "designer multi-layer cove ceiling with warm gold-lit trim",
        "materials": "real marble, brass trim, dark wood paneling",
        "palette": "rich dark wood tones with gold and brass metallic accents",
        "density": "furniture arranged in curated symmetrical conversation zones, restrained, not overfilled",
        "decor": "floor-to-ceiling heavy fabric curtains, framed art, symmetrical furniture placement",
        "structure_reminder": "still the same original room shape and window, do not enlarge or change the space",
    },
}

PRESERVE_STRUCTURE = (
    "same room structure, same walls, same windows, same doors, same ceiling height, "
    "same camera angle and perspective as the original photo, unchanged room dimensions"
)


def build_prompt(tier: str, room_description: str | None = None) -> str:
    spec = TIER_SPECS[tier]
    tokens = [f"{spec['label']} of the same room", PRESERVE_STRUCTURE]
    if room_description:
        tokens.append(room_description)
    tokens += [spec["paint"], spec["flooring"], spec["lighting_temp"], spec["ceiling"], spec["feature_wall"]]
    if spec.get("structure_reminder"):
        tokens.append(spec["structure_reminder"])
    tokens += [spec["materials"], spec["palette"], spec["density"], spec["decor"]]
    return ", ".join(tokens)


# Mage-Flow's pipe.edit() doesn't take a separate negative_prompt param (unlike SD1.5/Qwen) -
# it's a single instruction-conditioned model. Damage/luxury exclusions get folded into the
# positive instruction text instead, since there's no dedicated negative channel here.
NEGATIVE_FOLD_IN = {
    "economical": "no chandelier, no gold trim, no marble, not damaged or dirty, fully renovated",
    "mid": "no chandelier, no gold trim, no marble, not damaged or dirty, fully renovated",
    "premium": "not damaged or dirty, fully renovated",
}

for tier in TIER_SPECS:
    full_prompt = f"{build_prompt(tier)}. {NEGATIVE_FOLD_IN[tier]}."
    print(f"--- {tier} ---")
    print(full_prompt)
    print()

**Important caveat on the negative-prompt handling above**: the production pipeline's hard-won lesson (see CLAUDE.md) is that diffusion models don't reliably obey negation stated in a positive prompt - that's *why* Cloudflare's SD1.5 path uses a dedicated `negative_prompt` API parameter instead. Mage-Flow's `pipe.edit()` doesn't expose an equivalent parameter, so the cell above folds exclusions into the instruction text as a best-effort substitute - watch closely for the same chandelier-still-appears failure mode during review, since there's no negative channel here to fall back on.

## 6. Generate all 3 tiers

In [ ]:
results = {}

for tier in TIER_SPECS:
    print(f"Generating {tier}...")
    full_prompt = f"{build_prompt(tier)}. {NEGATIVE_FOLD_IN[tier]}."
    output = pipe.edit([full_prompt], [ROOM_PHOTO_PATH], steps=STEPS, cfg=5.0, max_size=MAX_SIZE)
    results[tier] = output[0]
    results[tier].save(f"mageflow_{tier}.png")

print("Done. Saved mageflow_economical.png, mageflow_mid.png, mageflow_premium.png")

## 7. View side by side

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

original = Image.open(ROOM_PHOTO_PATH)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(original)
axes[0].set_title("Original")
for ax, tier in zip(axes[1:], TIER_SPECS):
    ax.imshow(results[tier])
    ax.set_title(tier.capitalize())
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()